# DE9 - Niedersachsen

In [1]:
import os

import zipfile
from tqdm import tqdm
import pandas as pd
import pyproj
from camelsp import get_metadata

from camels_de1h import get_input_path, get_nuts_id_from_provider_id, Station1h

## Extract data

In [3]:
input_path = get_input_path("DE9")

# extract zip
if not (input_path / "Q_W").exists():
    os.makedirs(input_path / "Q_W")
    
    # extract the zip file
    with zipfile.ZipFile(input_path / "Q_W_Niedersachsen.zip", "r") as zip:
        zip.extractall(input_path / "Q_W")

    # metadata file is named incorrectly, rename it
    os.rename(input_path / "Q_W" / "2025-11-12_Stammdaten_Messstellen_Teillieferung_1_NLWKN.xlsx.xlsx",
              input_path / "Q_W" / "2025-11-12_Stammdaten_Messstellen_Teillieferung_1_NLWKN.xlsx")
        
    print("Data extracted.")
else:
    print("Data already extracted.")

Data already extracted.


## Parse data

Data comes from different sources, so formats and file names differ.  
Most files have the station ID in the beginning of the file name, some have the station name in the file name.  

The metadata .xlsx file is corrupt, we have to skip the first rows.

**Timezone: ???** -> for now I assume MEZ / UTC+1

In [4]:
raw_meta_all = pd.read_excel(input_path / "Q_W" / "2025-11-12_Stammdaten_Messstellen_Teillieferung_1_NLWKN.xlsx", skiprows=91, engine="openpyxl")
raw_meta_all["Messstelle Nr."] = raw_meta_all["Messstelle Nr."].astype(str)

ids = raw_meta_all["Messstelle Nr."].values

raw_meta_all

/home/alexd/miniconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Messstelle Nr.,Messstelle,East,North,Rechts 3,Hoch 3,Erfassungsgüte Koordinaten [m],Betreiber,Betrieb ab,Betrieb bis,...,Meldewesen,Pegelart,Schreibpegel,Messwertaufnehmer,Messwertübertragung,Querposition,Datenart Q,Datenart W,Einzugsgebiet [km²],Pegelnullpunkt [NN+m]
0,3863101,Neuburlage,32403567,5878144,3403595.0,5880043.0,NaN,NLWKN Betriebsstelle Aurich,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,ja,ja,62.0,-5.00
1,3882101,Aschhausen,32437925,5896233,3437970.0,5898160.0,NaN,NLWKN Betriebsstelle Brake-Oldenburg,1977-11-01,NaT,...,NaN,NaN,NaN,NaN,NaN,links,ja,ja,26.7,2.31
2,3882113,Harbern,32436679,5878729,NaN,NaN,NaN,NLWKN Betriebsstelle Brake-Oldenburg,1989-11-01,NaT,...,NaN,NaN,NaN,NaN,NaN,links,ja,ja,61.3,5.00
3,3884115,Ihorst,32423320,5901086,3423230.0,5903280.0,10.0,NLWKN Betriebsstelle Brake-Oldenburg,1982-11-01,NaT,...,NaN,NaN,NaN,NaN,NaN,rechts,ja,ja,51.9,-0.03
4,3886107,Südgeorgsfehn OP,32417163,5900908,NaN,NaN,NaN,NLWKN Betriebsstelle Aurich,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,ja,ja,32.0,-5.02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86,5952127,Jehrden,32567401,5916460,3567500.0,5918390.0,NaN,NLWKN Betriebsstelle Lüneburg,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,rechts,ja,ja,408.0,5.38
87,5958103,Langeloh,32552357,5903205,3552450.0,5905130.0,NaN,NLWKN Betriebsstelle Lüneburg,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,rechts,ja,ja,41.0,37.04
88,5958112,Emmen,32547889,5915530,3547980.0,5917460.0,NaN,NLWKN Betriebsstelle Lüneburg,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,links,ja,ja,184.0,11.47
89,9371101,Roggenstede,32399628,5944937,3399657.0,5946897.0,NaN,NLWKN Betriebsstelle Aurich,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,nein,ja,16.0,-5.00


In [5]:
print(f"{len(ids)} stations in metadata.")

files_q = list((input_path / "Q_W" / "01_Abflüsse").iterdir())
files_w = list((input_path / "Q_W" / "02_Wasserstände").iterdir())

print(f"{len(files_q)} files in 01_Abflüsse")
print(f"{len(files_w)} files in 02_Wasserstände")

91 stations in metadata.
82 files in 01_Abflüsse
86 files in 02_Wasserstände


Check data availability

In [6]:
only_q = 0
only_w = 0
both = 0
no_data = 0

for id in ids:
    file_q, file_w = None, None
    
    # get the station name from the metadata
    station_name = raw_meta_all.loc[raw_meta_all["Messstelle Nr."] == id, "Messstelle"].values[0]

    # get the q and w files for the station
    try:
        file_q = [f for f in files_q if id in f.name or station_name in f.name][0]
    except IndexError:
        pass
    try:
        file_w = [f for f in files_w if id in f.name or station_name in f.name][0]
    except IndexError:
        pass

    # count data availability
    if file_q and file_w:
        both += 1
    if file_q and not file_w:
        only_q += 1
    if file_w and not file_q:
        only_w += 1
    if not file_q and not file_w:
        no_data += 1

print(f"Stations with both Q and W data: {both}")
print(f"Stations with only Q data: {only_q}")
print(f"Stations with only W data: {only_w}")
print(f"Stations with no data: {no_data}")

Stations with both Q and W data: 76
Stations with only Q data: 5
Stations with only W data: 8
Stations with no data: 2


In [7]:
camelsp_meta_all = get_metadata()
camelsp_meta_all[camelsp_meta_all["provider_id"].isin(ids)]

,camels_id,provider_id,camels_path,nuts_lvl2,federal_state,gauge_name,waterbody_name,gauge_elevation,area,x,...,q_extent_years,w_extent_years,q_w_pearson,q_w_spearman,merit_hydro_available,q_more_than_10_years,merit_area_greater_5_smaller_15000,merit_completely_within_germany,merit_difference_to_reported_area_smaller_10_percent,merit_difference_to_reported_area_smaller_20_percent
1218,DE910520,3863101,./DE9/DE910520/DE910520_data.csv,DE9,Niedersachsen,Neuburlage,Burlage-Langholter Tief,NaN,68.312096,4.157473e+06,...,37.189041,37.189041,0.864043,0.549381,True,True,True,True,False,False
1222,DE910570,3882101,./DE9/DE910570/DE910570_data.csv,DE9,Niedersachsen,Aschhausen,Halfsteder Bäke,NaN,26.941689,4.192067e+06,...,38.189041,38.189041,0.897518,0.878349,True,True,True,True,True,True
1223,DE910580,3884115,./DE9/DE910580/DE910580_data.csv,DE9,Niedersachsen,Ihorst,Große Norderbäke,NaN,52.215598,4.177430e+06,...,15.175342,15.175342,0.893261,0.823569,True,True,True,True,False,True
1225,DE910600,3926104,./DE9/DE910600/DE910600_data.csv,DE9,Niedersachsen,Bagband,Bagbander Tief,NaN,13.527288,4.161524e+06,...,37.189041,37.189041,0.917415,0.813025,True,True,True,True,False,False
1231,DE910660,4536104,./DE9/DE910660/DE910660_data.csv,DE9,Niedersachsen,Holzminden,Hasselbach,NaN,15.628576,4.286072e+06,...,36.189041,36.189041,0.828696,0.732386,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1395,DE912440,5958112,./DE9/DE912440/DE912440_data.csv,DE9,Niedersachsen,Emmen,Este,NaN,184.595076,4.302372e+06,...,61.205479,61.205479,0.795718,0.650172,True,True,True,True,True,True
1425,DE912810,9421111,./DE9/DE912810/DE912810_data.csv,DE9,Niedersachsen,Neuenburg,Zeteler Tief,NaN,28.719626,4.184936e+06,...,35.186301,35.186301,0.532242,0.385923,True,True,True,True,True,True
1426,DE912850,4887101,./DE9/DE912850/DE912850_data.csv,DE9,Niedersachsen,Koldingen,Leine,NaN,4958.909671,4.307642e+06,...,NaN,34.189041,NaN,NaN,True,False,True,True,True,True
1427,DE912860,4948130,./DE9/DE912860/DE912860_data.csv,DE9,Niedersachsen,Tietjens Hütte,Hamme,NaN,462.359364,4.242146e+06,...,NaN,66.128767,NaN,NaN,True,False,True,True,True,True


In [10]:
def parse_station_data(id: str, aggregate_hourly: bool) -> pd.DataFrame | None:
    # get the station name from the metadata
    station_name = raw_meta_all.loc[raw_meta_all["Messstelle Nr."] == id, "Messstelle"].values[0]

    file_q, file_w = None, None

    # get the q and w files for the station
    try:
        file_q = [f for f in files_q if id in f.name or station_name in f.name][0]
    except IndexError:
        pass
    try:
        file_w = [f for f in files_w if id in f.name or station_name in f.name][0]
    except IndexError:
        pass

    if not file_q and not file_w:
        print(f"No data for station {id} - {station_name}")
        return None

    # read the data
    data_q = pd.read_csv(file_q, encoding="latin1", sep=";", decimal=",", na_values=["---"]) if file_q else None
    data_w = pd.read_csv(file_w, encoding="latin1", sep=";", decimal=",", na_values=["---"], dtype={"tagKommentare": str}) if file_w else None

    # possible column names from different sources
    q_cols = ["Wert [Kubikmeter pro Sekunde]", "Durchfluss [m³/s]"]
    w_cols = ["Wert [Zentimeter]", "Relativer Wert [cm]"]

    # Q: merge date and time columns if necessary and get discharge colunm
    if data_q is not None:
        if "Datum" in data_q.columns and "Uhrzeit" in data_q.columns and "Datum/Uhrzeit" not in data_q.columns:
            data_q["Datum/Uhrzeit"] = data_q["Datum"].astype(str) + " " + data_q["Uhrzeit"].astype(str)
            data_q = data_q.drop(columns=["Datum", "Uhrzeit"])

        # rename columns
        if not any(q_col in data_q.columns for q_col in q_cols):
            raise ValueError(f"No discharge column found for station {id} - {station_name}\{data_q.columns}")
        else:
            for q_col in q_cols:
                if q_col in data_q.columns:
                    data_q = data_q.rename(columns={q_col: "discharge_vol_obs", "Datum/Uhrzeit": "date"}, errors="raise")

        # select only relevant columns
        data_q = data_q[["date", "discharge_vol_obs"]]

    # W: merge date and time columns if necessary and get water level column
    if data_w is not None:
        if "Datum" in data_w.columns and "Uhrzeit" in data_w.columns and "Datum/Uhrzeit" not in data_w.columns:
            data_w["Datum/Uhrzeit"] = data_w["Datum"].astype(str) + " " + data_w["Uhrzeit"].astype(str)
            data_w = data_w.drop(columns=["Datum", "Uhrzeit"])

        # rename columns
        if not any(w_col in data_w.columns for w_col in w_cols):
            raise ValueError(f"No water level column found for station {id} - {station_name}\{data_w.columns}")
        else:
            for w_col in w_cols:
                if w_col in data_w.columns:
                    data_w = data_w.rename(columns={w_col: "water_level_obs", "Datum/Uhrzeit": "date"}, errors="raise")

        # select only relevant columns
        data_w = data_w[["date", "water_level_obs"]]

    # merge q and w data
    if data_q is not None and data_w is not None:
        data = pd.merge(data_q, data_w, left_on="date", right_on="date", how="outer", suffixes=("_q", "_w"))
        # check if all q or w are NaN
        if data["discharge_vol_obs"].isna().all():
            print(f"All discharge values are NaN for station {id} - {station_name}")
        if data["water_level_obs"].isna().all():
            print(f"All water level values are NaN for station {id} - {station_name}")

    elif data_q is not None:
        data = data_q
        data["water_level_obs"] = None
        if data["discharge_vol_obs"].isna().all():
            print(f"All discharge values are NaN for station {id} - {station_name}")

    elif data_w is not None:
        data = data_w
        data["discharge_vol_obs"] = None
        if data["water_level_obs"].isna().all():
            print(f"All water level values are NaN for station {id} - {station_name}")

    # reorder columns
    data = data[["date", "discharge_vol_obs", "water_level_obs"]]

    # parse date column
    data["date"] = pd.to_datetime(data["date"], format="%d.%m.%Y %H:%M:%S")

    # transform date time to UTC+0 (data is in UTC+1)
    data["date"] = data["date"] - pd.Timedelta(hours=1)
    data["date"] = data["date"].dt.tz_localize("UTC")

    if aggregate_hourly:
        data = data.set_index("date").resample("h", label="right", closed="right").mean().reset_index()

    # delete rows in the beginning where both columns are NaN
    while not data.empty and data.iloc[0][["discharge_vol_obs", "water_level_obs"]].isna().all():
        data = data.iloc[1:].reset_index(drop=True)

    # print a message if both columns are NaN in the first row
    # if data.iloc[0][["discharge_vol_obs", "water_level_obs"]].isna().all():
    #     print(f"First row contains NaN for both Q and W for station {id} - {station_name}")

    return data

parse_station_data("4833101", aggregate_hourly=False)

,date,discharge_vol_obs,water_level_obs
0,2003-12-31 23:15:00+00:00,20.1,359.0
1,2003-12-31 23:30:00+00:00,20.1,359.0
2,2003-12-31 23:45:00+00:00,20.1,359.0
3,2004-01-01 00:00:00+00:00,19.0,358.0
4,2004-01-01 00:15:00+00:00,19.0,358.0
...,...,...,...
736410,2024-12-31 21:45:00+00:00,23.0,362.0
736411,2024-12-31 22:00:00+00:00,22.4,362.0
736412,2024-12-31 22:15:00+00:00,23.4,362.0
736413,2024-12-31 22:30:00+00:00,23.6,362.0


In [11]:
parse_station_data("3882113", aggregate_hourly=True)

,date,discharge_vol_obs,water_level_obs
0,2007-11-01 12:00:00+00:00,0.19900,170.0
1,2007-11-01 13:00:00+00:00,0.22600,170.0
2,2007-11-01 14:00:00+00:00,0.22600,170.0
3,2007-11-01 15:00:00+00:00,0.22600,170.0
4,2007-11-01 16:00:00+00:00,0.22600,170.0
...,...,...,...
150487,2024-12-31 19:00:00+00:00,0.61025,179.0
150488,2024-12-31 20:00:00+00:00,0.61100,179.0
150489,2024-12-31 21:00:00+00:00,0.61100,179.0
150490,2024-12-31 22:00:00+00:00,0.61100,179.0


In [12]:
# make geometry from East / North coordinates
easting, northing = raw_meta_all["East"], raw_meta_all["North"]

transformer = pyproj.Transformer.from_crs("EPSG:4647", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(easting.values, northing.values)

raw_meta_all["lon"] = lon
raw_meta_all["lat"] = lat

In [13]:
def check_metadata_consistency(camelsp_meta, raw_meta, id: str) -> dict:
    """
    Check consistency of metadata fields across different metadata sources.
    Waterbody field is only compared between raw and camelsp.
    
    """
    inconsistencies = {}
    
    fields = {
        "provider_id": {
            "camelsp": "provider_id",
            "raw": "Messstelle Nr.",
        },
        "gauge_name": {
            "camelsp": "gauge_name",
            "raw": "Messstelle"
        },
        # "waterbody": {
        #     "camelsp": "waterbody_name",
        #     "raw": "river_name"
        # },
        "area": {
            "camelsp": "area",
            "raw": "Einzugsgebiet [km²]"
        },
        # "gauge_elevation": {
        #     "camelsp": "gauge_elevation",
        #     "raw": "Pegelnullpunkt [NN+m]"
        # },
        "lat": {
            "camelsp": "lat",
            "raw": "lat"
        },
        "lon": {
            "camelsp": "lon",
            "raw": "lon"
        }
    }

    if id not in camelsp_meta["provider_id"].values:
        print(f"ID {id} not found in camelsp metadata.")
        return None
    else:
        camelsp_row = camelsp_meta[camelsp_meta["provider_id"] == id].iloc[0]
        raw_row = raw_meta[raw_meta["Messstelle Nr."] == id].iloc[0]

        for field, sources in fields.items():
            camelsp_value = camelsp_row[sources["camelsp"]]
            raw_value = raw_row[sources["raw"]]

            if field in ["area", "gauge_elevation"]:
                # allow small difference due to rounding
                if not abs(camelsp_value - raw_value) < 2:
                    inconsistencies[field] = (camelsp_value, raw_value)
            elif field in ["lat", "lon"]:
                # allow small difference due to rounding
                if not abs(camelsp_value - raw_value) < 0.1:
                    inconsistencies[field] = (camelsp_value, raw_value)
            else:
                if camelsp_value != raw_value:
                    inconsistencies[field] = (camelsp_value, raw_value)
    
    return inconsistencies

inconsistencies_all = {}

for id in ids:
    inconsistencies_all[id] = check_metadata_consistency(camelsp_meta_all, raw_meta_all, id)

# show inconsistencies_all with at least one inconsistency
{ id: inc for id, inc in inconsistencies_all.items() if inc }

ID 3882113 not found in camelsp metadata.
ID 3886107 not found in camelsp metadata.
ID 3942102 not found in camelsp metadata.
ID 3983102 not found in camelsp metadata.
ID 4892110 not found in camelsp metadata.
ID 4896119 not found in camelsp metadata.
ID 4914104 not found in camelsp metadata.
ID 4965130 not found in camelsp metadata.
ID 5934145 not found in camelsp metadata.
ID 5952115 not found in camelsp metadata.


{'3863101': {'area': (68.31209567190001, 62.0)},
 '3926104': {'area': (13.5272878125, 48.0)},
 '4569105': {'area': (473.595918406, 470.0)},
 '4569106': {'area': (506.860812438, 509.0)},
 '4589101': {'area': (99.061732875, 102.0)},
 '4833101': {'area': (3524.69232642, 3533.0)},
 '4848111': {'area': (183.340124891, 179.0)},
 '4849104': {'area': (818.078166766, 812.0)},
 '4863115': {'area': (200.482191891, 198.0)},
 '4869108': {'area': (726.6455586879999, 738.0)},
 '4872119': {'area': (239.972570406, 242.0)},
 '4885154': {'area': (3454.57810559, 3463.0)},
 '4887101': {'area': (4958.909670939999, 4979.0)},
 '4888121': {'area': (157.945798984, 154.0)},
 '4888139': {'area': (568.7146850629999, 558.0)},
 '4899108': {'area': (9.63284634375, 95.0)},
 '4945108': {'area': (892.726858484, 908.0)},
 '4965142': {'area': (1710.21183281, 1714.0)},
 '5934140': {'area': (1302.2159805899998, 1300.0)},
 '9371101': {'area': (31.6379723125, 16.0)},
 '9421111': {'area': (28.719625875, nan)}}

Area can differ quite a lot between old and new metadata, from information online, the new metadata seems to be more accurate. So **we use the new metadata area** and maybe update CAMELS-DE later.

No river name in new metadata -> use old metadata river name, if available.  

We have no gauge elevation in CAMELS-DE metadata, but got gauge elevations in the new metadata -> could be added to CAMELS-DE later.

In [17]:
# Define lookup table for missing river names
river_name_mapping = {
    "3882113": "Vehne",
    "3886107": "Südgeorgsfehnkanal",
    "3942102": "Fehntjer Tief",
    "3983102": "Neues Greetsieler Sieltief",
    "4892110": "Meiße",
    "4896119": "Wölpe",
    "4914104": "Goldbach",
    "4965130": "Hunte",
    "5934145": "Jeetzel",
    "5952115": "Schmale Aue"
}

In [18]:
for id in tqdm(ids):
    # get the gauge_id from the nuts_mapping, add it if it does not exist
    gauge_id = get_nuts_id_from_provider_id(id, "DE9", add_missing=True)

    # parse data for the station
    data = parse_station_data(id, aggregate_hourly=True)

    if data is None:
        print(f"No data for station {id}, skipping.")
        continue

    # get the metadata of the station
    camelsp_meta = camelsp_meta_all[camelsp_meta_all["camels_id"] == gauge_id]
    raw_meta = raw_meta_all[raw_meta_all["Messstelle Nr."] == id]

    # both metadata sources available
    if not camelsp_meta.empty and not raw_meta.empty:
        # get river name
        waterbody_name = river_name_mapping.get(id, None)
        if waterbody_name is None:
            waterbody_name = camelsp_meta["waterbody_name"].values[0]

        if waterbody_name is None:
            print(f"Warning: No waterbody name for station {id} in camelsp metadata.")

        # set metadata values: mainly from camelsp, area and gauge elevation from new metadata
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": camelsp_meta["gauge_name"].values[0],
            "waterbody_name": camelsp_meta["waterbody_name"].values[0],
            "federal_state": camelsp_meta["federal_state"].values[0],
            "gauge_lon": camelsp_meta["lon"].values[0],
            "gauge_lat": camelsp_meta["lat"].values[0],
            "gauge_easting": camelsp_meta["x"].values[0],
            "gauge_northing": camelsp_meta["y"].values[0],
            "gauge_elev_metadata": raw_meta["Pegelnullpunkt [NN+m]"].values[0],
            "area_metadata": raw_meta["Einzugsgebiet [km²]"].values[0],
            "part_of_camelsp": True,
        }
    elif camelsp_meta.empty and not raw_meta.empty:
        # parse the location
        easting_raw, northing_raw = raw_meta["East"].values[0], raw_meta["North"].values[0]

        # from EPSG:4647 to EPSG:3035
        transformer = pyproj.Transformer.from_crs("epsg:4647", "epsg:3035", always_xy=True)
        easting, northing = transformer.transform(easting_raw, northing_raw)

        # from EPSG:4647 to EPSG:4326
        transformer = pyproj.Transformer.from_crs("EPSG:4647", "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(easting_raw, northing_raw)

        # get river name from mapping if available
        waterbody_name = river_name_mapping.get(id, None)

        if waterbody_name is None:
            print(f"Warning: No waterbody name for station {id} in raw metadata and mapping.")

        # if the station is not in the camelsp metadata, use the raw metadata
        station_meta = {
            "gauge_id": gauge_id,
            "provider_id": id,
            "gauge_name": raw_meta["Messstelle"].values[0],
            "waterbody_name": waterbody_name,
            "federal_state": "Niedersachsen",
            "gauge_lon": lon,
            "gauge_lat": lat,
            "gauge_easting": easting,
            "gauge_northing": northing,
            "gauge_elev_metadata": raw_meta["Pegelnullpunkt [NN+m]"].values[0],
            "area_metadata": raw_meta["Einzugsgebiet [km²]"].values[0],
            "part_of_camelsp": False,
        }
    else:
        raise ValueError(f"Station {id} has no metadata.")
    
    # initialize the station
    station = Station1h(gauge_id)

    # save time series data
    station.save_data(data)

    # save metadata
    station.save_metadata(**station_meta)
    
    # save raw metadata
    station.save_raw_metadata(raw_meta)

  0%|          | 0/91 [00:00<?, ?it/s]

 40%|███▉      | 36/91 [04:38<05:14,  5.72s/it]

No data for station 4886183 - Borsumer Paß
No data for station 4886183, skipping.


 71%|███████▏  | 65/91 [08:10<05:39, 13.08s/it]

No data for station 4948130 - Tietjens Hütte
No data for station 4948130, skipping.


100%|██████████| 91/91 [12:34<00:00,  8.29s/it]
